### Pré-processamento do Benchmark Dataset (Tsinghua Lab) para o participante S1

Este notebook organiza os dados do participante S1 no formato esperado pelo MNE, cria os objetos Epochs, aplica os filtros temporais e salva os resultados para o notebook de visualização.

In [ ]:
# Carregar os dados

import numpy as np
import pandas as pd

data = np.load('data/output/S1_pre_pros.npy')

print(data.dtype)
print(data.shape)

channels = pd.read_csv('data/input/64-channels.loc', sep=r'\s+', header=None, usecols=[3])[3].tolist() # Lê 64 canais do arquivo '64-channels.loc'
fs = 250
electrode_index, time_points, target_index, block_index = data.shape

**Reorganização do vetor**

Para isso, cada par (bloco, alvo) virou uma *epoch* de formato [64, 1500], sendo 6 blocos x 40 alvos (e trials) = 240 epochs. 

In [ ]:
# Reordena o array para [bloco, alvo, canal, tempo] e junta bloco e alvo em um único índice.
import mne

epochs_all = data.transpose(3, 2, 0, 1)
print(epochs_all.shape)

epochs_all = epochs_all.reshape(block_index * target_index, electrode_index, time_points)
print(epochs_all.shape)

**Verificação**

Como cada época deve ser idêntica ao trecho correspondente do array original, o trecho abaixo compara a época 0 (bloco 0, alvo 0), a época 21 (bloco 1, alvo 1) e a época 239 (bloco 5, alvo 39). Como o *reshape* percorre os blocos e, dentro de cada bloco, os alvos, a *epoch* de índice `b * 40 + t` correponde a `(bloco b, alvo t)`.

In [ ]:
target_labels = np.tile(np.arange(target_index), block_index)

print(np.array_equal(epochs_all[0], data[:, :, 0, 0]))
print(np.array_equal(epochs_all[41], data[:, :, 1, 1]))
print(np.array_equal(epochs_all[239], data[:, :, 39, 5]))
print(target_labels[0:44])
print(target_labels.shape)

**Objeto info**

O info reúne a frequência de amostragem de 250 Hz e os nomes dos 64 canais lidos do arquivo de posições, todos declarados como canais de EEG.

In [ ]:
sfreq = fs
ch_names = channels

info = mne.create_info(ch_names, sfreq=sfreq, ch_types='eeg')
info

**Criação do EpochsArray**

Com o info pronto, foram criados os eventos, que dizem ao MNE qual alvo foi olhado em cada epoch. Como o MNE assume que o sinal está em volts e os dados estão em microvolts, estes foram multiplicados por 1 × 10⁻⁶ ao criar o objeto. 

O tmin foi definido como -0,5 s porque cada epoch começa 500 ms antes do estímulo, de modo que o zero do eixo de tempo marca o início do estímulo. O resultado é um único objeto com as 240 epochs, cada uma com 1500 pontos, indo de -0,5 s até cerca de 5,5 s.

In [ ]:
pre_stimulus_time = 0.5

# Eventos: índice da epoch, valor anterior e código do alvo (os alvos viram códigos para poder selecionar as epochs por nome nas etapas seguintes)
events = np.column_stack((
    np.arange(len(target_labels)),
    np.zeros(len(target_labels), dtype=int),
    target_labels
))

# Nome de cada código de alvo
event_id = {}
for target in range(target_index):
    event_id['alvo_' + str(target)] = target

# Objeto Epochs em volts, com tempo zero no início do estímulo
mne_epochs = mne.EpochsArray(epochs_all * 1e-6, info, events, tmin=-pre_stimulus_time, event_id=event_id)

print(mne_epochs)

**Aplicação de filtros**

Para filtrar, foi utilizado o método filter do próprio objeto Epochs do MNE porque ele já aplica o filtro em cada canal de todas as epochs, sem deslocar a fase do sinal.

Como as frequências dos alvos vão de 8 a 15,8 Hz e o segundo harmônico chega a 31,6 Hz, foi aplicado um passa-faixa de 6 a 35 Hz, que mantém as frequências fundamentais e os segundos harmônicos e remove a deriva lenta e o ruído de frequências altas. O ruído da touca de 50 Hz já fica fora dessa faixa, e o filtro notch também já havia sido aplicado na gravação.

O filtro foi aplicado em uma cópia, para manter o objeto sem filtro e poder comparar os dois no domínio da frequência. 

In [ ]:
# Passa-faixa de 6 a 35 Hz em uma cópia do objeto
mne_filtered = mne_epochs.copy().filter(l_freq=6., h_freq=35.)

# Salva os dois objetos para o notebook de visualização
mne_epochs.save('data/output/S1_epo.fif', overwrite=True)
mne_filtered.save('data/output/S1_filtered_epo.fif', overwrite=True)